In [4]:
%pip install -q fastapi uvicorn httpx

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:

import sys
from pathlib import Path

project_dir = Path(
    "/content/drive/MyDrive/Portfolio_Project/Abstract_Classifier"
)

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

In [3]:
from pathlib import Path

print("Current working directory:", Path.cwd())
print("Project directory:", project_dir)

# Ensure the tests folder exists in your project.
(project_dir / "tests").mkdir(parents=True, exist_ok=True)

# Make relative paths resolve from the project directory.
%cd $project_dir

print("Tests folder exists:", Path("tests").is_dir())

Current working directory: /content
Project directory: /content/drive/MyDrive/Portfolio_Project/Abstract_Classifier
/content/drive/MyDrive/Portfolio_Project/Abstract_Classifier
Tests folder exists: True


In [16]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-v"],
    cwd=project_dir,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr:
    print(result.stderr)

result.check_returncode()

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/Portfolio_Project/Abstract_Classifier
plugins: platformdirs-4.14.0, typeguard-4.6.0, langsmith-0.14.15, anyio-4.15.1
collecting ... collected 11 items

tests/test_api.py::test_health PASSED                                    [  9%]
tests/test_api.py::test_predict_returns_ordered_results PASSED           [ 18%]
tests/test_api.py::test_predict_rejects_invalid_input[payload0] PASSED   [ 27%]
tests/test_api.py::test_predict_rejects_invalid_input[payload1] PASSED   [ 36%]
tests/test_api.py::test_predict_rejects_invalid_input[payload2] PASSED   [ 45%]
tests/test_core.py::test_position_cnn_checkpoint_round_trip PASSED       [ 54%]
tests/test_core.py::test_relative_position PASSED                        [ 63%]
tests/test_data.py::test_collate_position_batch PASSED                

In [14]:
import importlib

import pandas as pd
from fastapi.testclient import TestClient

import abstract_classifier.api as api_module

api_module = importlib.reload(api_module)

artifact_path = (
    project_dir / "artifacts" / "abstract_classifier_v1.pt"
)

prediction_app = api_module.create_app(model_path=artifact_path)

payload = {
    "sentences": [
        "Effective treatments for persistent fatigue remain limited.",
        "This study aimed to evaluate an exercise programme.",
        "We randomly assigned 350 patients to exercise or usual care.",
        "Fatigue scores were measured at baseline and after treatment.",
        "Patients in the exercise group reported lower fatigue scores.",
        "The programme may help reduce persistent fatigue.",
    ]
}

with TestClient(prediction_app) as client:
    health_response = client.get("/health")
    print("Health:", health_response.json())

    response = client.post("/predict", json=payload)

    print("Prediction HTTP status:", response.status_code)
    response.raise_for_status()

    display(pd.DataFrame(response.json()["predictions"]))

    # Also check that an empty sentence is rejected.
    invalid_response = client.post(
        "/predict",
        json={"sentences": ["   "]},
    )

    print("Invalid-input HTTP status:", invalid_response.status_code)
    assert invalid_response.status_code == 422

Health: {'status': 'ok', 'model_loaded': True}
Prediction HTTP status: 200


,sentence_number,relative_position,sentence,prepared_sentence,predicted_label
0,0,0.0,Effective treatments for persistent fatigue re...,Effective treatments for persistent fatigue re...,BACKGROUND
1,1,0.2,This study aimed to evaluate an exercise progr...,This study aimed to evaluate an exercise progr...,OBJECTIVE
2,2,0.4,We randomly assigned 350 patients to exercise ...,We randomly assigned 350 patients to exercise ...,METHODS
3,3,0.6,Fatigue scores were measured at baseline and a...,Fatigue scores were measured at baseline and a...,METHODS
4,4,0.8,Patients in the exercise group reported lower ...,Patients in the exercise group reported lower ...,RESULTS
5,5,1.0,The programme may help reduce persistent fatigue.,The programme may help reduce persistent fatig...,CONCLUSIONS


Invalid-input HTTP status: 422


Writing tests/test_api.py


In [12]:
import inspect
import torch

from abstract_classifier.inference import predict_raw_abstract

artifact_path = project_dir / "artifacts" / "abstract_classifier_v1.pt"

bundle = torch.load(
    artifact_path,
    map_location="cpu",
    weights_only=True,
)

print("Bundle keys:", list(bundle))
print("Model configuration:", bundle.get("config"))
print("Input contract:", bundle.get("input_contract"))
print("Inference signature:", inspect.signature(predict_raw_abstract))

Bundle keys: ['format_version', 'model_class', 'model_state_dict', 'vocab', 'label_to_id', 'pad_id', 'config', 'selection', 'input_contract']
Model configuration: {'model': 'CNN + position', 'seed': 42, 'embedding_dim': 64, 'num_filters': 32, 'kernel_size': 3, 'padding': 1, 'batch_size': 64, 'drop_last': False, 'learning_rate': 0.001, 'tokenizer': 'lowercase_then_whitespace_split', 'position_feature': 'sentence_number / max(total_sentences - 1, 1)'}
Input contract: {'input': 'Complete abstract as an ordered list of non-empty sentences', 'tokenizer': 'lowercase_then_whitespace_split', 'position': 'sentence_number / max(total_sentences - 1, 1)', 'optional_raw_adapter': 'limited_punctuation_spacing_v1', 'automatic_sentence_splitting': False}
Inference signature: (sentences, *, model, vocab, id_to_label)


In [11]:

from importlib.metadata import version

for package in ["fastapi", "uvicorn", "httpx"]:
    print(f"{package}=={version(package)}")

fastapi==0.141.1
uvicorn==0.54.0
httpx==0.28.1
